# 09 — Predicción de facturación diaria, largo plazo (sin memoria reciente)

**Motivación:** el modelo de la Fase 2 (`05_prediccion_facturacion_diaria_definitivo.ipynb`) usa variables de memoria (`facturacion_7d_antes`, medias móviles...) que dan mucha señal, pero que solo existen de verdad para el día siguiente al último dato conocido. Para fechas mucho más lejanas (p.ej. varios meses), rellenar esas columnas con un valor aproximado no es honesto: el modelo aprendió a darles peso real, no a que sean un valor inventado.

**Este notebook entrena un modelo alternativo, pensado explícitamente para horizontes largos: usa solo variables que se pueden conocer con meses de antelación** (calendario, festivos, eventos, meteorología si se aporta, reservas ya confirmadas) — nunca lo que pasó en los días/semanas anteriores. Se espera, y se confirma más abajo, que su error sea mayor que el del modelo de corto plazo: es el coste de quitar la señal más fuerte que había. La comparación honesta entre ambos es el resultado más importante de este notebook, no solo el modelo en sí.

Reutiliza la ingeniería de variables ya escrita en `ml/features.py` (Fase 2) en vez de recalcularla aquí — misma fuente de verdad que usa la aplicación.

In [1]:
import warnings
warnings.filterwarnings('ignore')

import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from ml.features import build_revenue_feature_table, REVENUE_FEATURE_COLS

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import TimeSeriesSplit, RandomizedSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib

RANDOM_STATE = 42
MODELS_DIR = PROJECT_ROOT / 'results' / 'models'
RESULTS_DIR = PROJECT_ROOT / 'results' / 'forecasting_largo_plazo'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
print('Project root:', PROJECT_ROOT)

Project root: C:\Users\CandelaGB\Desktop\TFM anita\TFM-Hosteleria-AI


## 1. Datos: mismas filas que el notebook 05, sin las columnas de memoria

`build_revenue_feature_table()` (de `ml/features.py`) construye exactamente las mismas 57 columnas que ve el modelo de corto plazo. Aquí nos quedamos solo con las 39 que no dependen de facturación pasada (todo lo que empieza por `facturacion_` se descarta). Al no depender de lags, **no hace falta descartar ninguna fila por falta de histórico** — se pueden usar los 242 días completos, uno más que el notebook 05.

In [2]:
MEMORIA_COLS = [c for c in REVENUE_FEATURE_COLS if c.startswith('facturacion_')]
FEATURE_COLS = [c for c in REVENUE_FEATURE_COLS if c not in MEMORIA_COLS]
print(f'Columnas de memoria descartadas ({len(MEMORIA_COLS)}): {MEMORIA_COLS}')
print(f'\nColumnas del modelo de largo plazo ({len(FEATURE_COLS)}): {FEATURE_COLS}')

df = build_revenue_feature_table()
X = df[FEATURE_COLS].copy()
y = df['facturacion'].copy()
fechas = df['fecha'].copy()
print(f'\nFilas disponibles: {len(X)} (notebook 05 usaba {len(X)-1} tras descartar la primera fila sin lag)')

Columnas de memoria descartadas (18): ['facturacion_1d_antes', 'facturacion_3d_antes', 'facturacion_7d_antes', 'facturacion_14d_antes', 'facturacion_21d_antes', 'facturacion_28d_antes', 'facturacion_media_3d', 'facturacion_media_7d', 'facturacion_std_7d', 'facturacion_total_7d', 'facturacion_media_14d', 'facturacion_std_14d', 'facturacion_total_14d', 'facturacion_media_28d', 'facturacion_std_28d', 'facturacion_total_28d', 'facturacion_media_90d', 'facturacion_tendencia_7_28']

Columnas del modelo de largo plazo (39): ['dia_semana', 'num_dia', 'mes', 'es_festivo', 'festivo_nombre', 'tiene_evento', 'intensidad_evento', 'impacto_evento', 'direccion_evento', 'categoria_evento', 'cat_evento', 'temperature_max', 'temperature_min', 'temperature_mean', 'precipitation_mm', 'precipitation_hours', 'wind_speed_max', 'sunshine_duration_h', 'es_fin_de_semana', 'es_lunes', 'reservas_anticipadas', 'comensales_anticipados', 'grupos_grandes_anticipados', 'antelacion_media_dias', 'anio', 'semana_anio', '


Filas disponibles: 242 (notebook 05 usaba 241 tras descartar la primera fila sin lag)


## 2. Split train/test cronológico (80/20, igual criterio que el notebook 05)

In [3]:
split = int(len(X) * 0.8)
X_train, X_test = X.iloc[:split].copy(), X.iloc[split:].copy()
y_train, y_test = y.iloc[:split].copy(), y.iloc[split:].copy()
fechas_train, fechas_test = fechas.iloc[:split], fechas.iloc[split:]

print('TRAIN:', fechas_train.min().date(), '->', fechas_train.max().date(), f'({len(X_train)} días)')
print('TEST :', fechas_test.min().date(), '->', fechas_test.max().date(), f'({len(X_test)} días)')

TRAIN: 2025-10-02 -> 2026-05-13 (193 días)
TEST : 2026-05-14 -> 2026-07-09 (49 días)


## 3. Modelos: Ridge y Random Forest

Mismos dos tipos de modelo que dominaron el ranking del notebook 05 (Random Forest fue el ganador allí, Ridge el punto de referencia lineal). Búsqueda de hiperparámetros más modesta que en el notebook original (menos combinaciones): este modelo es un complemento explicativo, no el modelo principal del TFM.

In [4]:
categorical_cols = X_train.select_dtypes(include=['object', 'category', 'bool']).columns.tolist()
numeric_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()

categorical_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
])
numeric_linear = Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())])
numeric_tree = Pipeline([('imputer', SimpleImputer(strategy='median'))])

preprocessor_linear = ColumnTransformer([('num', numeric_linear, numeric_cols), ('cat', categorical_pipe, categorical_cols)])
preprocessor_tree = ColumnTransformer([('num', numeric_tree, numeric_cols), ('cat', categorical_pipe, categorical_cols)])

tscv = TimeSeriesSplit(n_splits=5)

pipe_ridge = Pipeline([('preprocessor', preprocessor_linear), ('model', Ridge(random_state=RANDOM_STATE))])
search_ridge = RandomizedSearchCV(
    pipe_ridge, {'model__alpha': [0.1, 0.3, 1, 3, 10, 30, 100, 300]},
    n_iter=8, cv=tscv, scoring='neg_mean_absolute_error', random_state=RANDOM_STATE, n_jobs=-1,
)
search_ridge.fit(X_train, y_train)

pipe_rf = Pipeline([('preprocessor', preprocessor_tree), ('model', RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=1))])
search_rf = RandomizedSearchCV(
    pipe_rf,
    {'model__n_estimators': [200, 400], 'model__max_depth': [None, 4, 6, 10],
     'model__min_samples_leaf': [1, 2, 4], 'model__max_features': [0.5, 0.7, 1.0]},
    n_iter=10, cv=tscv, scoring='neg_mean_absolute_error', random_state=RANDOM_STATE, n_jobs=-1,
)
search_rf.fit(X_train, y_train)

print(f'Ridge  MAE_CV = {-search_ridge.best_score_:.2f} €  params: {search_ridge.best_params_}')
print(f'RF     MAE_CV = {-search_rf.best_score_:.2f} €  params: {search_rf.best_params_}')

Ridge  MAE_CV = 536.68 €  params: {'model__alpha': 30}
RF     MAE_CV = 530.96 €  params: {'model__n_estimators': 200, 'model__min_samples_leaf': 2, 'model__max_features': 0.7, 'model__max_depth': 10}


## 4. Evaluación en test — comparación honesta con baselines y con el modelo de corto plazo

In [5]:
def smape(y_true, y_pred):
    denom = np.abs(y_true) + np.abs(y_pred)
    m = denom != 0
    return np.mean(2 * np.abs(y_pred[m] - y_true[m]) / denom[m]) * 100

candidatos = {'Ridge (largo plazo)': search_ridge.best_estimator_, 'Random Forest (largo plazo)': search_rf.best_estimator_}
resultados = []
for nombre, modelo in candidatos.items():
    pred = modelo.predict(X_test)
    resultados.append({
        'modelo': nombre, 'MAE_€': mean_absolute_error(y_test, pred),
        'RMSE_€': mean_squared_error(y_test, pred) ** 0.5, 'R2': r2_score(y_test, pred),
        'sMAPE_%': smape(y_test.values, pred),
    })

baseline_naive = y.shift(1).iloc[split:].values
mask_b = ~np.isnan(baseline_naive)
resultados.append({
    'modelo': 'Baseline (día anterior)', 'MAE_€': mean_absolute_error(y_test[mask_b], baseline_naive[mask_b]),
    'RMSE_€': mean_squared_error(y_test[mask_b], baseline_naive[mask_b]) ** 0.5,
    'R2': r2_score(y_test[mask_b], baseline_naive[mask_b]), 'sMAPE_%': smape(y_test[mask_b].values, baseline_naive[mask_b]),
})

# Referencia: MAE test del modelo de CORTO plazo (notebook 05, Random Forest, ya evaluado allí)
MAE_CORTO_PLAZO = 457.19
resultados.append({'modelo': 'Modelo corto plazo (referencia, notebook 05)', 'MAE_€': MAE_CORTO_PLAZO, 'RMSE_€': None, 'R2': None, 'sMAPE_%': None})

comparacion = pd.DataFrame(resultados).sort_values('MAE_€')
display(comparacion)

mejor_nombre = min(candidatos, key=lambda n: mean_absolute_error(y_test, candidatos[n].predict(X_test)))
mejor_modelo = candidatos[mejor_nombre]
print(f'\nMejor modelo de largo plazo: {mejor_nombre}')

,modelo,MAE_€,RMSE_€,R2,sMAPE_%
3,"Modelo corto plazo (referencia, notebook 05)",457.190000,NaN,NaN,NaN
1,Random Forest (largo plazo),494.347110,620.344061,0.729086,17.386241
0,Ridge (largo plazo),863.671240,1014.203257,0.275870,29.228892
2,Baseline (día anterior),1102.155918,1448.871987,-0.477837,37.440555



Mejor modelo de largo plazo: Random Forest (largo plazo)


## 5. Conclusión

Se espera (y hay que comprobar en la tabla anterior) que el MAE de este modelo sea claramente peor que los 457€ del modelo de corto plazo — esa diferencia **es** el precio de no tener historial reciente, no un error de entrenamiento. Este modelo no sustituye al de la Fase 2: se usa únicamente cuando se predice una fecha demasiado lejana para tener memoria reciente real (ver `ml/revenue_predictor.py`).

## 6. Guardado del modelo

In [6]:
ruta_modelo = MODELS_DIR / 'mejor_modelo_facturacion_diaria_largo_plazo.joblib'
joblib.dump(mejor_modelo, ruta_modelo)
comparacion.to_csv(RESULTS_DIR / 'comparacion_facturacion.csv', index=False)
print(f'Modelo guardado en: {ruta_modelo}')

Modelo guardado en: C:\Users\CandelaGB\Desktop\TFM anita\TFM-Hosteleria-AI\results\models\mejor_modelo_facturacion_diaria_largo_plazo.joblib